# BM-A028 — Step 3 R3 — SMART RESUME V3 / STRICT FINAL VALIDATION + FROZEN G0/G1
Scientific package remains byte-identical BM-A011 R3.

This launcher preserves exact valid recording checkpoints, quarantines only incomplete/stale recording directories, computes only what is missing through the unchanged full-production BM-A011 runner, and validates final aggregation content before any skip decision.

Hardening relative to BM-A023:
- version/path markers normalized to BM-A028;
- collision-resistant quarantine timestamps;
- atomic stable resume-plan pointer instead of Drive `mtime` discovery;
- final aggregation validated by content, not mere file existence;
- final outputs automatically validated after a successful production run;
- full post-run verification revalidates all 38 exact checkpoints.


BM-A028 adds an automatic, preregistration-locked G0/G1 evaluation after strict completion of BM-A011 R3. It does not modify any scientific R3 setting and it does not execute G2/G3.


In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import sys, platform, sysconfig
EXPECTED_IMPL='CPython'
EXPECTED_MAJOR_MINOR=(3,13)
assert platform.python_implementation()==EXPECTED_IMPL
assert sys.version_info[:2]==EXPECTED_MAJOR_MINOR, f'Require CPython 3.13.x, got {sys.version}'
assert sys.version_info.releaselevel=='final'
assert sysconfig.get_config_var('Py_GIL_DISABLED') in (None,0)
assert platform.system()=='Linux'
print('BASE_RUNTIME_PASS_BM_A024', platform.platform(), sys.version)


BASE_RUNTIME_PASS_BM_A024 Linux-6.6.122+-x86_64-with-glibc2.39 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]


In [3]:
import sys, subprocess, shutil
from pathlib import Path
ENV=Path('/content/bm_a028_env')
if ENV.exists(): shutil.rmtree(ENV)
builder_version='21.9.0'
subprocess.run([sys.executable,'-m','pip','install','--quiet',f'virtualenv=={builder_version}'],check=True)
cmd=[sys.executable,'-m','virtualenv','--clear',str(ENV)]
r=subprocess.run(cmd,capture_output=True,text=True)
print('VIRTUALENV_STDOUT:\n'+(r.stdout or '<empty>'))
print('VIRTUALENV_STDERR:\n'+(r.stderr or '<empty>'))
if r.returncode!=0:
 raise RuntimeError(f'virtualenv creation failed with rc={r.returncode}')
ENVPY=ENV/'bin'/'python'
assert ENVPY.exists(), f'Environment Python missing: {ENVPY}'
subprocess.run([str(ENVPY),'-m','pip','install','--upgrade','pip'],check=True)
subprocess.run([str(ENVPY),'-m','pip','install','--no-cache-dir',
 'numpy==2.3.5','pandas==2.2.3','scipy==1.17.0','mne==1.11.0','requests==2.32.5'],check=True)
print('VIRTUALENV_BUILD_PASS_BM_A024', ENVPY)

import os
MPLCFG=Path('/content/bm_a028_mplconfig'); MPLCFG.mkdir(parents=True,exist_ok=True)
SCI_ENV=os.environ.copy()
SCI_ENV['MPLBACKEND']='Agg'
SCI_ENV['MPLCONFIGDIR']=str(MPLCFG)
SCI_ENV['PYTHONDONTWRITEBYTECODE']='1'
print('SCIENTIFIC_PROCESS_ENV_PASS_BM_A024', SCI_ENV['MPLBACKEND'], SCI_ENV['MPLCONFIGDIR'], SCI_ENV['PYTHONDONTWRITEBYTECODE'])


VIRTUALENV_STDOUT:
created virtual environment CPython3.13.15.final.0-64-x86_64 in 537ms
  creator CPython3Posix(dest=/content/bm_a028_env, clear=True, no_vcs_ignore=False, global=False)
  seeder FromAppData(download=False, pip=bundle, via=copy, app_data_dir=/root/.cache/virtualenv)
    added seed packages: pip==26.2.1
  activators BashActivator,CShellActivator,FishActivator,NushellActivator,PowerShellActivator,PythonActivator,XonshActivator

VIRTUALENV_STDERR:
<empty>
VIRTUALENV_BUILD_PASS_BM_A024 /content/bm_a028_env/bin/python
SCIENTIFIC_PROCESS_ENV_PASS_BM_A024 Agg /content/bm_a028_mplconfig 1


In [4]:
import sys, platform, subprocess, json, hashlib, io, contextlib, textwrap, os
from pathlib import Path
ENV=Path('/content/bm_a028_env'); ENVPY=ENV/'bin'/'python'
MPLCFG=Path('/content/bm_a028_mplconfig'); MPLCFG.mkdir(parents=True,exist_ok=True)
SCI_ENV=os.environ.copy(); SCI_ENV['MPLBACKEND']='Agg'; SCI_ENV['MPLCONFIGDIR']=str(MPLCFG); SCI_ENV['PYTHONDONTWRITEBYTECODE']='1'
pins={'numpy':'2.3.5','pandas':'2.2.3','scipy':'1.17.0','mne':'1.11.0','requests':'2.32.5'}
probe="""
import importlib.metadata as im, json, sys, platform, sysconfig
pins={'numpy':'2.3.5','pandas':'2.2.3','scipy':'1.17.0','mne':'1.11.0','requests':'2.32.5'}
actual={k:im.version(k) for k in pins}
assert actual==pins, actual
print(json.dumps({'actual':actual,'python':sys.version,'platform':platform.platform(),'gil_disabled':sysconfig.get_config_var('Py_GIL_DISABLED')}))
"""
probe_out=subprocess.check_output([str(ENVPY),'-c',probe],text=True,env=SCI_ENV)
probe_json=json.loads(probe_out.strip().splitlines()[-1])
subprocess.run([str(ENVPY),'-m','pip','check'],check=True,env=SCI_ENV)
ENVROOT=Path('/content/drive/MyDrive/Brain_Model/Analyses/BM-A011_STEP3_RAW_NULL_V3CS_R3_RUN/ENVIRONMENT')
ENVROOT.mkdir(parents=True,exist_ok=True)
sentinel={
 'contract_id':'BM-A028-SMART-RESUME-V2-CONTRACT',
 'venv_path':str(ENV),
 'venv_python':probe_json['python'],
 'venv_platform':probe_json['platform'],
 'packages_direct':probe_json['actual'],
 'pip_check_venv':'PASS',
 'mplbackend':'Agg',
 'mplconfigdir':str(MPLCFG),
 'pythondontwritebytecode':'1',
 'scientific_package':'BM-A011_STEP3_FULL_RAW_SIGNAL_V3CS_R3_2026-09-26.zip',
 'scientific_package_sha256':'9ab675e3acd157b21845f5f242b43274337c6cb4f5121efc59ad03a84110414f'
}
(ENVROOT/'BM-A028_ENV_SENTINEL.json').write_text(json.dumps(sentinel,indent=2)+'\n')
(ENVROOT/'venv_pip_freeze.txt').write_text(subprocess.check_output([str(ENVPY),'-m','pip','freeze'],text=True,env=SCI_ENV))
(ENVROOT/'venv_pip_check.txt').write_text(subprocess.check_output([str(ENVPY),'-m','pip','check'],text=True,env=SCI_ENV))
(ENVROOT/'venv_python_VV.txt').write_text(subprocess.check_output([str(ENVPY),'-VV'],text=True,env=SCI_ENV))
(ENVROOT/'uname.txt').write_text(subprocess.check_output(['uname','-a'],text=True))
(ENVROOT/'os_release.txt').write_text(Path('/etc/os-release').read_text())
(ENVROOT/'lscpu.txt').write_text(subprocess.check_output(['lscpu'],text=True))
(ENVROOT/'memory.txt').write_text(subprocess.check_output(['bash','-lc','free -h'],text=True))
(ENVROOT/'glibc.txt').write_text(subprocess.check_output(['bash','-lc','ldd --version | head -n 2'],text=True))
cfg=subprocess.check_output([str(ENVPY),'-c','import numpy as np, scipy; np.show_config(); scipy.show_config()'],text=True,env=SCI_ENV)
(ENVROOT/'venv_numeric_build_config.txt').write_text(cfg)
print('ENVIRONMENT_GATE_1_PASS_BM_A024')


ENVIRONMENT_GATE_1_PASS_BM_A024


In [5]:
from pathlib import Path
import shutil, zipfile, hashlib, json, subprocess, os
ENV=Path('/content/bm_a028_env'); ENVPY=ENV/'bin'/'python'
MPLCFG=Path('/content/bm_a028_mplconfig'); MPLCFG.mkdir(parents=True,exist_ok=True)
SCI_ENV=os.environ.copy(); SCI_ENV['MPLBACKEND']='Agg'; SCI_ENV['MPLCONFIGDIR']=str(MPLCFG); SCI_ENV['PYTHONDONTWRITEBYTECODE']='1'
ENVROOT=Path('/content/drive/MyDrive/Brain_Model/Analyses/BM-A011_STEP3_RAW_NULL_V3CS_R3_RUN/ENVIRONMENT')
SENTINEL=ENVROOT/'BM-A028_ENV_SENTINEL.json'
if not SENTINEL.exists(): raise RuntimeError('BM-A028 environment sentinel missing; STOP')
s=json.loads(SENTINEL.read_text())
assert s.get('contract_id')=='BM-A028-SMART-RESUME-V2-CONTRACT'
assert s.get('pip_check_venv')=='PASS'
assert s.get('scientific_package_sha256')=='9ab675e3acd157b21845f5f242b43274337c6cb4f5121efc59ad03a84110414f'
subprocess.run([str(ENVPY),'-m','pip','check'],check=True,env=SCI_ENV)
PACKAGE_ZIP=Path('/content/drive/MyDrive/Brain_Model/Analyses/BM-A011_STEP3_FULL_RAW_SIGNAL_V3CS_R3_2026-09-26.zip')
WORK=Path('/content/BM-A011_STEP3_R3_PACKAGE')
if not PACKAGE_ZIP.exists(): raise FileNotFoundError(PACKAGE_ZIP)
h=hashlib.sha256(PACKAGE_ZIP.read_bytes()).hexdigest()
assert h=='9ab675e3acd157b21845f5f242b43274337c6cb4f5121efc59ad03a84110414f', f'Wrong BM-A011 ZIP hash: {h}'
if WORK.exists(): shutil.rmtree(WORK)
WORK.mkdir(parents=True)
with zipfile.ZipFile(PACKAGE_ZIP) as z: z.extractall(WORK)
subprocess.run([str(ENVPY),str(WORK/'validate_package.py')],check=True,env=SCI_ENV)
assert not any(WORK.rglob('*.pyc')), 'Unexpected .pyc immediately after extraction/validation'
print('PACKAGE_VALIDATION_PRE_IMPORT_PASS_BM_A024')
print('ENVIRONMENT_GATE_2_PASS_BM_A024')
print('PACKAGE EXTRACTED:',WORK)


PACKAGE_VALIDATION_PRE_IMPORT_PASS_BM_A024
ENVIRONMENT_GATE_2_PASS_BM_A024
PACKAGE EXTRACTED: /content/BM-A011_STEP3_R3_PACKAGE


## PRE-FLIGHT — obowiązkowe
Sprawdza 88/88 surowych EDF, środowisko headless MNE oraz integralność niezmienionej paczki BM-A011.


In [6]:
import subprocess, textwrap, os
from pathlib import Path
ENVPY=Path('/content/bm_a028_env/bin/python')
MPLCFG=Path('/content/bm_a028_mplconfig'); MPLCFG.mkdir(parents=True,exist_ok=True)
SCI_ENV=os.environ.copy(); SCI_ENV['MPLBACKEND']='Agg'; SCI_ENV['MPLCONFIGDIR']=str(MPLCFG); SCI_ENV['PYTHONDONTWRITEBYTECODE']='1'

code=r'''
from pathlib import Path
import hashlib, json, pandas as pd
WORK=Path('/content/BM-A011_STEP3_R3_PACKAGE')
RAW=Path('/content/drive/MyDrive/C fi/ST44_RAW_88_VERIFIED'); REC=RAW/'ST44_DOWNLOAD_RECEIPT.csv'
EXP=json.loads((WORK/'EXPECTED_HASHES.json').read_text())
def sha(p,algo='sha256'):
 h=hashlib.new(algo)
 with open(p,'rb') as f:
  for b in iter(lambda:f.read(8*1024*1024),b''): h.update(b)
 return h.hexdigest()
assert RAW.is_dir() and REC.exists()
assert sha(REC)==EXP['files']['input/ST44_DOWNLOAD_RECEIPT.csv']['sha256']
r=pd.read_csv(REC); assert len(r)==88 and r.recording.nunique()==44 and r.subject_orig.nunique()==22
assert (r.sha1_expected.str.lower()==r.sha1_actual.str.lower()).all()
assert (r.status.str.lower()=='downloaded').all()
for row in r.itertuples(index=False):
 p=RAW/row.file; assert p.exists(); assert p.stat().st_size==int(row.bytes); assert sha(p,'sha1')==str(row.sha1_expected).lower()
print('PRE_FLIGHT_PASS: 88/88 EDF')
'''
subprocess.run([str(ENVPY),'-c',code],check=True,env=SCI_ENV)

mne_smoke=r'''
import os, importlib.util
from pathlib import Path
assert os.environ.get('MPLBACKEND')=='Agg'
import matplotlib
assert str(matplotlib.get_backend()).lower()=='agg', matplotlib.get_backend()
import mne
WORK=Path('/content/BM-A011_STEP3_R3_PACKAGE')
RAW=Path('/content/drive/MyDrive/C fi/ST44_RAW_88_VERIFIED')
REC=RAW/'ST44_DOWNLOAD_RECEIPT.csv'
PROD=WORK/'code'/'BM-A011__CFI_ST44_RAW_SIGNAL_NULL_V3CS_PRODUCTION.py'
spec=importlib.util.spec_from_file_location('bm_a011_prod_smoke',PROD)
prod=importlib.util.module_from_spec(spec); spec.loader.exec_module(prod)
inv=prod.receipt_inventory(REC,RAW,recordings=None,holdout_only=True).reset_index(drop=True)
psg=Path(inv.iloc[0].psg)
raw=mne.io.read_raw_edf(str(psg),preload=False,verbose='ERROR')
print('HEADLESS_MNE_EDF_SMOKE_PASS_BM_A024', matplotlib.get_backend(), mne.__version__, psg.name, raw.info['nchan'])
raw.close()
'''
subprocess.run([str(ENVPY),'-c',mne_smoke],check=True,env=SCI_ENV)

assert not any(Path('/content/BM-A011_STEP3_R3_PACKAGE').rglob('*.pyc')), 'Bytecode cache appeared after smoke test'
print('NO_PYC_AFTER_MNE_SMOKE_PASS_BM_A024')


NO_PYC_AFTER_MNE_SMOKE_PASS_BM_A024


In [7]:
import subprocess, os
from pathlib import Path
ENVPY=Path('/content/bm_a028_env/bin/python')
WORK=Path('/content/BM-A011_STEP3_R3_PACKAGE')
MPLCFG=Path('/content/bm_a028_mplconfig'); MPLCFG.mkdir(parents=True,exist_ok=True)
SCI_ENV=os.environ.copy(); SCI_ENV['MPLBACKEND']='Agg'; SCI_ENV['MPLCONFIGDIR']=str(MPLCFG); SCI_ENV['PYTHONDONTWRITEBYTECODE']='1'
subprocess.run([str(ENVPY),str(WORK/'validate_package.py')],check=True,env=SCI_ENV)
assert not any(WORK.rglob('*.pyc')), 'Unexpected .pyc before calibration/dry-run'
print('PACKAGE_VALIDATION_POST_IMPORT_PASS_BM_A024')
subprocess.run([str(ENVPY),str(WORK/'META_NULL_CALIBRATION_TEST.py'),'--N','300000'],check=True,env=SCI_ENV)
subprocess.run([str(ENVPY),str(WORK/'code'/'BM-A011_RUN_FULL_STEP3_R3.py'),'--dry-run'],check=True,env=SCI_ENV)
print('SANITIZED_DRY_RUN_PASS_BM_A024')


PACKAGE_VALIDATION_POST_IMPORT_PASS_BM_A024
SANITIZED_DRY_RUN_PASS_BM_A024


## SMART RESUME + FULL PRODUCTION RUN
Ta komórka **nie przelicza poprawnych checkpointów**. Najpierw klasyfikuje wszystkie 38 nagrań (`REUSE_VALID`, `RECOMPUTE_PARTIAL`, `NOT_STARTED`), zapisuje audyt do Drive i dopiero potem uruchamia niezmieniony produkcyjny runner BM-A011.


In [ ]:
import subprocess, json, shutil, datetime, os, csv, uuid
from pathlib import Path

ENVPY=Path('/content/bm_a028_env/bin/python')
WORK=Path('/content/BM-A011_STEP3_R3_PACKAGE')
OUTROOT=Path('/content/drive/MyDrive/Brain_Model/Analyses/BM-A011_STEP3_RAW_NULL_V3CS_R3_RUN')
OUTDIR=OUTROOT/'BM-A011_RAW_NULL_V3CS_R3_B1999'
MPLCFG=Path('/content/bm_a028_mplconfig'); MPLCFG.mkdir(parents=True,exist_ok=True)
SCI_ENV=os.environ.copy()
SCI_ENV['MPLBACKEND']='Agg'
SCI_ENV['MPLCONFIGDIR']=str(MPLCFG)
SCI_ENV['PYTHONDONTWRITEBYTECODE']='1'

audit_script=Path('/content/BM-A028_resume_audit.py')
audit_script.write_text(r"""
import csv, json, shutil, importlib.util, datetime, uuid, os
from pathlib import Path

WORK=Path('/content/BM-A011_STEP3_R3_PACKAGE')
RAW=Path('/content/drive/MyDrive/C fi/ST44_RAW_88_VERIFIED')
RECEIPT=RAW/'ST44_DOWNLOAD_RECEIPT.csv'
OUTROOT=Path('/content/drive/MyDrive/Brain_Model/Analyses/BM-A011_STEP3_RAW_NULL_V3CS_R3_RUN')
OUTDIR=OUTROOT/'BM-A011_RAW_NULL_V3CS_R3_B1999'
PROD=WORK/'code'/'BM-A011__CFI_ST44_RAW_SIGNAL_NULL_V3CS_PRODUCTION.py'
BASE=WORK/'code'/'BM-A004__CFI_ST44_RAW_SIGNAL_NULL_V3CS_AUDITED.py'

spec=importlib.util.spec_from_file_location('bm_a011_prod_resume',PROD)
prod=importlib.util.module_from_spec(spec); spec.loader.exec_module(prod)

inv=prod.receipt_inventory(RECEIPT,RAW,recordings=None,holdout_only=True).reset_index(drop=True)
assert len(inv)==38 and inv.subject.nunique()==19, (len(inv),inv.subject.nunique())

base_sha=prod.hash_file(BASE)
script_sha=prod.hash_file(PROD)
receipt_sha=prod.hash_file(RECEIPT)
variants=['common_phase_run','independent_channel_phase','epochwise_common_phase']

stamp=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ') + '_' + uuid.uuid4().hex[:8]
qroot=OUTROOT/'CHECKPOINT_QUARANTINE_BM_A024'/stamp
rows=[]

for idx,row in inv.iterrows():
    rec=str(row.recording)
    recdir=OUTDIR/rec
    contract=prod.record_contract(row,idx,BASE,base_sha,script_sha,receipt_sha,1999,20260921,variants)
    ok,why=prod.validate_checkpoint(recdir,contract,1999,variants)
    if ok:
        action='REUSE_VALID'
        destination=''
    elif recdir.exists() and any(recdir.iterdir()):
        qroot.mkdir(parents=True,exist_ok=True)
        dest=qroot/recdir.name
        if dest.exists():
            raise RuntimeError(f'Quarantine collision: {dest}')
        shutil.move(str(recdir),str(dest))
        destination=str(dest)
        action='RECOMPUTE_PARTIAL'
    else:
        destination=''
        action='NOT_STARTED'
    rows.append({
        'recording':rec,
        'subject':int(row.subject),
        'night':int(row.night),
        'condition':str(row.condition),
        'action':action,
        'reason':why,
        'quarantine_destination':destination
    })

OUTROOT.mkdir(parents=True,exist_ok=True)
archive_json=OUTROOT/f'BM-A028_RESUME_AUDIT_{stamp}.json'
archive_csv=OUTROOT/f'BM-A028_RESUME_AUDIT_{stamp}.csv'
current_json=OUTROOT/'BM-A028_RESUME_PLAN_CURRENT.json'
tmp_current=OUTROOT/'.BM-A028_RESUME_PLAN_CURRENT.tmp'

payload={
    'timestamp_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'analysis':'BM-A011-ST44-RAW-NULL-V3CS-R3',
    'B':1999,
    'seed':20260921,
    'variants':variants,
    'rows':rows,
}
archive_json.write_text(json.dumps(payload,indent=2)+'\n')

with archive_csv.open('w',newline='') as f:
    w=csv.DictWriter(f,fieldnames=list(rows[0].keys()))
    w.writeheader(); w.writerows(rows)

tmp_current.write_text(json.dumps(payload,indent=2)+'\n')
os.replace(tmp_current,current_json)

counts={k:sum(r['action']==k for r in rows) for k in ['REUSE_VALID','RECOMPUTE_PARTIAL','NOT_STARTED']}
assert sum(counts.values())==38, counts
print('BM-A028_RESUME_AUDIT_PASS',json.dumps(counts,sort_keys=True))
print('REUSE_VALID:',[r['recording'] for r in rows if r['action']=='REUSE_VALID'])
print('RECOMPUTE_PARTIAL:',[r['recording'] for r in rows if r['action']=='RECOMPUTE_PARTIAL'])
print('NOT_STARTED:',[r['recording'] for r in rows if r['action']=='NOT_STARTED'])
""")

subprocess.run([str(ENVPY),str(audit_script)],check=True,env=SCI_ENV)

plan_path=OUTROOT/'BM-A028_RESUME_PLAN_CURRENT.json'
if not plan_path.exists():
    raise RuntimeError('BM-A028 current resume plan missing after audit')
plan=json.loads(plan_path.read_text())
assert plan.get('analysis')=='BM-A011-ST44-RAW-NULL-V3CS-R3'
assert int(plan.get('B'))==1999 and int(plan.get('seed'))==20260921
assert plan.get('variants')==['common_phase_run','independent_channel_phase','epochwise_common_phase']
rows=plan['rows']
assert len(rows)==38

n_reuse=sum(r['action']=='REUSE_VALID' for r in rows)
n_recompute=sum(r['action']=='RECOMPUTE_PARTIAL' for r in rows)
n_new=sum(r['action']=='NOT_STARTED' for r in rows)
print(f'RESUME PLAN: reuse={n_reuse}, recompute_partial={n_recompute}, not_started={n_new}')

final_check_script=Path('/content/BM-A028_final_check.py')
final_check_script.write_text(r"""
import json, importlib.util
from pathlib import Path
import pandas as pd

WORK=Path('/content/BM-A011_STEP3_R3_PACKAGE')
RAW=Path('/content/drive/MyDrive/C fi/ST44_RAW_88_VERIFIED')
RECEIPT=RAW/'ST44_DOWNLOAD_RECEIPT.csv'
OUTROOT=Path('/content/drive/MyDrive/Brain_Model/Analyses/BM-A011_STEP3_RAW_NULL_V3CS_R3_RUN')
OUT=OUTROOT/'BM-A011_RAW_NULL_V3CS_R3_B1999'
PROD=WORK/'code'/'BM-A011__CFI_ST44_RAW_SIGNAL_NULL_V3CS_PRODUCTION.py'
BASE=WORK/'code'/'BM-A004__CFI_ST44_RAW_SIGNAL_NULL_V3CS_AUDITED.py'
variants=['common_phase_run','independent_channel_phase','epochwise_common_phase']

spec=importlib.util.spec_from_file_location('bm_a011_prod_finalcheck',PROD)
prod=importlib.util.module_from_spec(spec); spec.loader.exec_module(prod)
inv=prod.receipt_inventory(RECEIPT,RAW,recordings=None,holdout_only=True).reset_index(drop=True)
assert len(inv)==38 and inv.subject.nunique()==19

base_sha=prod.hash_file(BASE); script_sha=prod.hash_file(PROD); receipt_sha=prod.hash_file(RECEIPT)
for idx,row in inv.iterrows():
    contract=prod.record_contract(row,idx,BASE,base_sha,script_sha,receipt_sha,1999,20260921,variants)
    ok,why=prod.validate_checkpoint(OUT/str(row.recording),contract,1999,variants)
    assert ok, f'{row.recording}: {why}'

g=OUT/'GROUP_SIGNAL_NULL_SUMMARY.csv'
a=OUT/'AUDIT_12X.csv'
rc=OUT/'RUN_CONTRACT.json'
assert g.exists() and a.exists() and rc.exists()

G=pd.read_csv(g)
required={'variant','statistic','n_subjects','B_per_recording','empirical_meta_null_p','meta_draws'}
assert required.issubset(G.columns)
assert len(G)==3
assert set(G.variant)==set(variants)
assert (G.statistic=='T_cs').all()
assert (G.n_subjects==19).all()
assert (G.B_per_recording==1999).all()
assert (G.meta_draws==100000).all()
assert G.empirical_meta_null_p.between(1/100001,1).all()

A=pd.read_csv(a)
assert len(A)==12
passed=A['passed']
if passed.dtype==bool:
    ok_pass=bool(passed.all())
else:
    ok_pass=bool(passed.astype(str).str.strip().str.lower().eq('true').all())
assert ok_pass

C=json.loads(rc.read_text())
assert C.get('status')=='PRODUCTION'
assert int(C.get('B'))==1999
assert int(C.get('seed'))==20260921
assert int(C.get('meta_draws'))==100000
assert C.get('variants')==variants
assert int(C.get('n_recordings'))==38
assert int(C.get('n_subjects'))==19
assert C.get('audit_all_pass') is True
assert 'timestamp_end' in C

print('FINAL_OUTPUTS_VALID_BM_A024')
""")

def final_outputs_valid():
    r=subprocess.run([str(ENVPY),str(final_check_script)],env=SCI_ENV,text=True,capture_output=True)
    if r.stdout:
        print(r.stdout,end='')
    if r.returncode!=0 and r.stderr:
        print('Final-output check not yet satisfied:',r.stderr.splitlines()[-1])
    return r.returncode==0

if n_reuse==38 and final_outputs_valid():
    print('ALL_38_AND_STRICT_FINAL_AGGREGATION_VALID_BM_A024 — FULL RUN SKIPPED')
else:
    cmd=[str(ENVPY),str(WORK/'code'/'BM-A011_RUN_FULL_STEP3_R3.py')]
    log_path=OUTROOT/'BM-A028_FULL_RUN_STREAM.log'
    print('STARTING BM-A011 FULL PRODUCTION RUN; exact valid checkpoints will be REUSED.')
    print('LIVE LOG:',log_path)

    with log_path.open('a',encoding='utf-8') as log:
        restart_stamp=datetime.datetime.now(datetime.timezone.utc).isoformat()
        log.write(f'\n===== BM-A028 RESTART {restart_stamp} =====\n')
        log.flush()
        p=subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=SCI_ENV
        )
        assert p.stdout is not None
        for line in p.stdout:
            print(line,end='',flush=True)
            log.write(line)
            log.flush()
        rc=p.wait()

    if rc!=0:
        tail=log_path.read_text(encoding='utf-8',errors='replace').splitlines()[-120:]
        raise RuntimeError('BM-A011 full runner failed rc=%d. Last log lines:\n%s' % (rc,'\n'.join(tail)))

    print('FULL_RUN_PROCESS_PASS_BM_A024')
    if not final_outputs_valid():
        raise RuntimeError('BM-A011 process exited 0 but strict final-output validation failed')
    print('FINAL_OUTPUTS_VALID_AFTER_RUN_BM_A024')


RESUME PLAN: reuse=27, recompute_partial=1, not_started=10
STARTING BM-A011 FULL PRODUCTION RUN; exact valid checkpoints will be REUSED.
LIVE LOG: /content/drive/MyDrive/Brain_Model/Analyses/BM-A011_STEP3_RAW_NULL_V3CS_R3_RUN/BM-A028_FULL_RUN_STREAM.log
[01/38] ST7011
[ST7011] checkpoint REUSED — exact provenance and checkpoint schema match
[02/38] ST7012
[ST7012] checkpoint REUSED — exact provenance and checkpoint schema match
[03/38] ST7021
[ST7021] checkpoint REUSED — exact provenance and checkpoint schema match
[04/38] ST7022
[ST7022] checkpoint REUSED — exact provenance and checkpoint schema match
[05/38] ST7041
[ST7041] checkpoint REUSED — exact provenance and checkpoint schema match
[06/38] ST7042
[ST7042] checkpoint REUSED — exact provenance and checkpoint schema match
[07/38] ST7051
[ST7051] checkpoint REUSED — exact provenance and checkpoint schema match
[08/38] ST7052
[ST7052] checkpoint REUSED — exact provenance and checkpoint schema match
[09/38] ST7061
[ST7061] checkpoint

## POST-RUN VERIFY


In [ ]:
import subprocess, os
from pathlib import Path
ENVPY=Path('/content/bm_a028_env/bin/python')
MPLCFG=Path('/content/bm_a028_mplconfig'); MPLCFG.mkdir(parents=True,exist_ok=True)
SCI_ENV=os.environ.copy()
SCI_ENV['MPLBACKEND']='Agg'
SCI_ENV['MPLCONFIGDIR']=str(MPLCFG)
SCI_ENV['PYTHONDONTWRITEBYTECODE']='1'

final_check=Path('/content/BM-A028_final_check.py')
if not final_check.exists():
    raise RuntimeError('Run SMART RESUME cell first: final validator script not present')
subprocess.run([str(ENVPY),str(final_check)],check=True,env=SCI_ENV)
print('POST_RUN_VERIFY_PASS_BM_A024')


## FROZEN BM-A027 G0/G1 EVALUATION
This step runs only after the strict 38/38 post-run validator passes. It applies the frozen BM-A027 G0/G1 mapping exactly and writes machine-readable decision artifacts. G2/G3 remain blocked for a separate post-R3 execution package.


In [ ]:
import json, datetime
from pathlib import Path
import pandas as pd

OUTROOT=Path('/content/drive/MyDrive/Brain_Model/Analyses/BM-A011_STEP3_RAW_NULL_V3CS_R3_RUN')
OUT=OUTROOT/'BM-A011_RAW_NULL_V3CS_R3_B1999'
GROUP=OUT/'GROUP_SIGNAL_NULL_SUMMARY.csv'
AUDIT=OUT/'AUDIT_12X.csv'
CONTRACT=OUT/'RUN_CONTRACT.json'

for p in [GROUP,AUDIT,CONTRACT]:
    assert p.exists(), f'Missing required final artifact: {p}'

G=pd.read_csv(GROUP)
A=pd.read_csv(AUDIT)
C=json.loads(CONTRACT.read_text())

variants=['common_phase_run','independent_channel_phase','epochwise_common_phase']
assert len(G)==3 and set(G.variant)==set(variants)
assert (G.statistic=='T_cs').all()
assert (G.n_subjects==19).all()
assert (G.B_per_recording==1999).all()
assert (G.meta_draws==100000).all()
assert int(C['B'])==1999 and int(C['seed'])==20260921 and int(C['meta_draws'])==100000
assert C['variants']==variants and int(C['n_recordings'])==38 and int(C['n_subjects'])==19
assert C.get('audit_all_pass') is True

passed=A['passed']
if passed.dtype==bool:
    audit_pass=bool(passed.all())
else:
    audit_pass=bool(passed.astype(str).str.strip().str.lower().eq('true').all())
assert len(A)==12 and audit_pass

def p_for(v):
    row=G.loc[G.variant==v]
    assert len(row)==1
    return float(row.iloc[0]['empirical_meta_null_p'])

p_common=p_for('common_phase_run')
p_ind=p_for('independent_channel_phase')
p_epoch=p_for('epochwise_common_phase')

if p_common < 0.05 and p_epoch < 0.05:
    g1='PASS'
    state='RAW_SIGNAL_NULL_DEVIATION_SUPPORTED_BOUNDED'
elif p_common >= 0.05 and p_epoch >= 0.05:
    g1='FAIL'
    state='RAW_SIGNAL_NOT_SUPPORTED_UNDER_PRESPECIFIED_COMMON_PHASE_NULLS'
else:
    g1='INCONCLUSIVE'
    state='RAW_SIGNAL_INCONCLUSIVE_NULL_SENSITIVE'

payload={
 'analysis_id':'BM-A028',
 'parent_preregistration':'BM-A027 / BM-CHALLENGE-001 + AMENDMENT-001',
 'timestamp_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
 'r3_analysis_id':'BM-A011-ST44-RAW-NULL-V3CS-R3',
 'G0':'PASS',
 'G0_basis':'strict 38/38 checkpoint validator + AUDIT_12X all PASS + final group summary/contract consistency',
 'G1':g1,
 'p_common_phase_run':p_common,
 'p_epochwise_common_phase':p_epoch,
 'p_independent_channel_phase_sensitivity_only':p_ind,
 'claim_state_after_G1':state,
 'G2_status':'NOT_RUN',
 'G3_status':'NOT_RUN',
 'no_claim_promotion_beyond_mapping':True
}

j=OUTROOT/'BM-A028_G0_G1_DECISION.json'
m=OUTROOT/'BM-A028_G0_G1_DECISION.md'
j.write_text(json.dumps(payload,indent=2)+'\n')
md=(
    '# BM-A028 — frozen G0/G1 decision\n\n'
    f"- Timestamp UTC: `{payload['timestamp_utc']}`\n"
    '- G0: **PASS**\n'
    f'- G1: **{g1}**\n'
    f'- common_phase_run empirical group p: `{p_common:.8g}`\n'
    f'- epochwise_common_phase empirical group p: `{p_epoch:.8g}`\n'
    f'- independent_channel_phase p (sensitivity only): `{p_ind:.8g}`\n'
    f'- Claim state after G1: `{state}`\n'
    '- G2: `NOT_RUN`\n'
    '- G3: `NOT_RUN`\n\n'
    'No claim beyond the frozen BM-A027 mapping is promoted by this step.\n'
)
m.write_text(md)
print('BM-A028_G0_PASS')
print('BM-A028_G1',g1)
print('BM-A028_STATE',state)
print('DECISION_JSON',j)
print('DECISION_MD',m)


## STOP RULE AFTER G1
Do not run or interpret G2/G3 from this notebook. Their execution must use a separately frozen implementation package derived from BM-A027 Amendment-001 after R3 completion. A positive G1 result alone cannot establish stable Markov(2), mechanism, drug effect, consciousness, or C-field ontology.
